# 01 Idea Validation (Canonical)

One end-to-end workflow proving that tool calls are executed by your local Python code through eXo-brain's deterministic path.

- Local smoke test works without `OPENAI_API_KEY`
- Live model run requires `OPENAI_API_KEY`

In [ ]:
import os
import pathlib
import random
import sys
import uuid
from typing import Optional

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))

_env = _root / ".env"
if _env.exists():
    try:
        from dotenv import load_dotenv
        load_dotenv(_env, override=False)
        print(f"Loaded .env from {_env}")
    except Exception as exc:
        print(f"Could not load .env via dotenv: {exc}")

from src.policies.middleware import DeterministicFirstPolicyMiddleware
from src.schemas.tool_io import RiskTier, ToolCallContext, ToolStatus
from src.tools.executor import DeterministicToolExecutor
from src.tools.registry import ToolDescriptor, ToolRegistry
from agents import Agent, ModelSettings, Runner, function_tool

HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))
print("OPENAI_API_KEY set:", HAS_KEY)

## Define deterministic local function

`operand3` is server-side secret and is never trusted from model input.

In [ ]:
SECRET_OPERAND3 = random.randint(1, 101)

def _calculate_result(operation: str, operand1: float, operand2: float, operand3: float = SECRET_OPERAND3) -> dict:
    if operation == "add":
        value = operand1 + operand2 + operand3
    elif operation == "subtract":
        value = operand1 - operand2 + operand3
    elif operation == "multiply":
        value = operand1 * operand2 + operand3
    elif operation == "divide":
        if operand2 == 0:
            raise ValueError("Cannot divide by zero")
        value = operand1 / operand2 + operand3
    else:
        raise ValueError(f"Unknown operation: {operation}")

    return {
        "operation": operation,
        "operand1": operand1,
        "operand2": operand2,
        "operand3": operand3,
        "result": value,
    }

print("SECRET_OPERAND3:", SECRET_OPERAND3)
print("Local smoke add(5,7):", _calculate_result("add", 5, 7)["result"])

## Wire registry + policy + deterministic executor

In [ ]:
registry = ToolRegistry()
registry.register(
    ToolDescriptor(
        name="calculate_result",
        handler=_calculate_result,
        risk_tier=RiskTier.LOW,
        is_state_changing=False,
    )
)
policy = DeterministicFirstPolicyMiddleware()
executor = DeterministicToolExecutor(registry=registry, policy=policy)
print("Registered tools:", registry.list_tools())

## Mirror tool schema for model, but inject server secret

In [ ]:
@function_tool
def calculate_result(operation: str, operand1: float, operand2: float, operand3: Optional[float] = None):
    print(f"[intercepted] model operand3={operand3!r}; server operand3={SECRET_OPERAND3}")
    call = ToolCallContext(
        schema_version="1.0",
        call_id=str(uuid.uuid4()),
        session_id="sess_idea",
        run_id="run_idea",
        job_id="job_idea",
        task_id="task_idea",
        agent_id="idea-agent",
        provider_id="openai",
        tool_name="calculate_result",
        arguments={
            "operation": operation,
            "operand1": operand1,
            "operand2": operand2,
            "operand3": SECRET_OPERAND3,
        },
        risk_tier=RiskTier.LOW,
        is_state_changing=False,
    )
    result = executor.execute(call)
    if result.status != ToolStatus.SUCCESS:
        raise ValueError(result.error.message or "tool call failed")
    payload = result.result.get("value", {}) if isinstance(result.result, dict) else {}
    out = payload.get("result", payload)
    print(f"[intercepted] returned result={out}")
    return out

## Single live workflow run (`OPENAI_API_KEY` required)

In [ ]:
if not HAS_KEY:
    print("Skip live run: OPENAI_API_KEY not set.")
else:
    instructions = (
        "You are a math assistant. Always call calculate_result for arithmetic. "
        "Use operand3=0 as placeholder; server controls true operand3."
    )
    agent = Agent(
        name="idea-agent",
        instructions=instructions,
        model="gpt-4o-mini",
        tools=[calculate_result],
        model_settings=ModelSettings(parallel_tool_calls=False),
    )

    question = "What is 5 plus 7?"
    print("USER:", question)
    run = await Runner.run(agent, question)
    print("AGENT:", run.final_output)

## Expected Output Checklist

- You see `[intercepted]` lines printed by local Python function
- Server secret operand is injected and differs from model placeholder
- Agent final response reflects returned deterministic result